# Handschrifterkennung mit MNIST — Analyse-Notebook

Dieses Notebook führt eine vollständige Analyse des MNIST-Datensatzes durch:

1. **Datenexploration** — MNIST laden, Samples visualisieren, Klassenverteilung prüfen
2. **Modelltraining** — MLPClassifier (scikit-learn) trainieren
3. **Evaluierung** — Confusion Matrix, Classification Report
4. **Fehleranalyse** — Falsch klassifizierte Bilder untersuchen

Basiert auf `mnist_analysis.py`.

## 1. Importe und Setup

In [ ]:
import gzip
import os
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier

# Für schönere Plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("✅ Alle Importe erfolgreich!")

## 2. MNIST-Daten laden

Der MNIST-Datensatz besteht aus 70.000 handgeschriebenen Ziffern (0–9):
- **60.000** Trainingsbilder
- **10.000** Testbilder

Jedes Bild ist 28×28 Pixel groß (784 Features) und wird als Graustufenwert [0,1] normalisiert.

Die Daten werden von Google Cloud Storage heruntergeladen und lokal gecached.

In [ ]:
def load_mnist():
    """Lädt den MNIST-Datensatz mit lokalem Caching.

    Returns:
        tuple: (X_train, y_train, X_test, y_test) als numpy-Arrays.
            X: float32 [0,1], y: uint8 [0-9]
    """
    cache_dir = os.path.join(os.path.dirname(__file__) or ".", ".mnist_cache")
    os.makedirs(cache_dir, exist_ok=True)

    files = {
        "train_images": "train-images-idx3-ubyte.gz",
        "train_labels": "train-labels-idx1-ubyte.gz",
        "test_images": "t10k-images-idx3-ubyte.gz",
        "test_labels": "t10k-labels-idx1-ubyte.gz",
    }
    base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"

    for fname in files.values():
        path = os.path.join(cache_dir, fname)
        if not os.path.exists(path):
            print(f"  Lade {fname}...")
            try:
                with urlopen(base_url + fname) as response, open(path, "wb") as f:
                    f.write(response.read())
            except Exception as e:
                raise RuntimeError(
                    f"Fehler beim Download von {fname}: {e}"
                ) from e

    def load_images(path):
        """Lädt MNIST-Bilddaten aus einer IDX-Datei."""
        with gzip.open(path, "rb") as f:
            data = np.frombuffer(f.read(), np.uint8, offset=16)
        return data.reshape(-1, 784).astype(np.float32) / 255.0

    def load_labels(path):
        """Lädt MNIST-Labeldaten aus einer IDX-Datei."""
        with gzip.open(path, "rb") as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)

    X_train = load_images(os.path.join(cache_dir, files["train_images"]))
    y_train = load_labels(os.path.join(cache_dir, files["train_labels"]))
    X_test = load_images(os.path.join(cache_dir, files["test_images"]))
    y_test = load_labels(os.path.join(cache_dir, files["test_labels"]))

    return X_train, y_train, X_test, y_test


print("📦 Lade MNIST...")
X_train, y_train, X_test, y_test = load_mnist()
print(f"   Train: {X_train.shape[0]:,} Bilder | Test: {X_test.shape[0]:,} Bilder")
print(f"   Dimension: {X_train.shape[1]} Features (28×28 Pixel)")

## 3. Datenexploration

### 3.1 Klassenverteilung
Prüfen, ob die Ziffern gleichmäßig verteilt sind.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trainingsdaten
unique_train, counts_train = np.unique(y_train, return_counts=True)
axes[0].bar(unique_train, counts_train, color="steelblue", edgecolor="white")
axes[0].set_title("Trainingsdaten — Klassenverteilung")
axes[0].set_xlabel("Ziffer")
axes[0].set_ylabel("Anzahl")
for i, c in zip(unique_train, counts_train):
    axes[0].text(i, c + 50, str(c), ha="center", fontsize=8)

# Testdaten
unique_test, counts_test = np.unique(y_test, return_counts=True)
axes[1].bar(unique_test, counts_test, color="coral", edgecolor="white")
axes[1].set_title("Testdaten — Klassenverteilung")
axes[1].set_xlabel("Ziffer")
axes[1].set_ylabel("Anzahl")
for i, c in zip(unique_test, counts_test):
    axes[1].text(i, c + 5, str(c), ha="center", fontsize=8)

plt.tight_layout()
plt.show()

### 3.2 Zufällige Samples
10 zufällige Trainingsbilder mit ihren Labels.

In [ ]:
def plot_samples(X, y, n=10):
    """Zeigt n zufällige MNIST-Bilder mit Labels."""
    idx = np.random.choice(len(X), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(n * 1.5, 2))
    for i, ax in enumerate(axes):
        ax.imshow(X[idx[i]].reshape(28, 28), cmap="gray")
        ax.set_title(f"Label: {y[idx[i]]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("📸 Zufällige Trainings-Samples:")
plot_samples(X_train, y_train, n=10)

### 3.3 Durchschnittsbilder pro Ziffer
Wie sieht die „durchschnittliche" handschriftliche Ziffer aus?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    avg_img = X_train[y_train == digit].mean(axis=0).reshape(28, 28)
    im = ax.imshow(avg_img, cmap="hot")
    ax.set_title(f"Ziffer {digit}")
    ax.axis("off")
plt.suptitle("Durchschnittsbilder pro Ziffer (Trainingsdaten)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Modelltraining: MLPClassifier

Wir trainieren ein **Multi-Layer Perceptron (MLP)** mit scikit-learn:
- **Architektur**: 2 Hidden Layer (128 → 64 Neuronen)
- **Aktivierung**: ReLU
- **Optimierer**: Adam
- **Epochen**: 20

Das MLP lernt, handgeschriebene Ziffern anhand der 784 Pixelwerte zu klassifizieren.

In [ ]:
print("🔧 Training: scikit-learn MLPClassifier...")
print(f"   Architektur: 784 → 128 → 64 → 10")

model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    max_iter=20,
    random_state=42,
    verbose=True,
)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"\n   Train-Accuracy: {train_acc:.4f}")
print(f"   Test-Accuracy:  {test_acc:.4f}")

y_pred = model.predict(X_test)

## 5. Evaluierung

### 5.1 Confusion Matrix
Zeigt, welche Ziffern wie oft verwechselt werden.

In [ ]:
def plot_confusion(y_true, y_pred, title="Confusion Matrix"):
    """Zeigt die Confusion-Matrix als Heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=[str(i) for i in range(10)],
                yticklabels=[str(i) for i in range(10)])
    ax.set_xlabel("Vorhergesagt")
    ax.set_ylabel("Tatsächlich")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("📊 Confusion Matrix:")
plot_confusion(y_test, y_pred, title="Confusion Matrix — MLPClassifier")

### 5.2 Classification Report
Precision, Recall und F1-Score pro Ziffer.

In [ ]:
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

## 6. Fehleranalyse

### 6.1 Falsch klassifizierte Bilder
Welche Bilder werden vom Modell falsch erkannt? Hier sehen wir die problematischsten Fälle.

In [ ]:
def plot_misclassified(X, y_true, y_pred, n=10):
    """Zeigt falsch klassifizierte Bilder."""
    errors = np.where(y_true != y_pred)[0]
    if len(errors) == 0:
        print("Keine Fehler! 🎉")
        return
    n_show = min(n, len(errors))
    idx = np.random.choice(errors, n_show, replace=False)

    n_cols = min(n_show, 5)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes_flat = axes.flat if hasattr(axes, "flat") else axes.flatten()

    for i, ax in enumerate(axes_flat):
        if i < n_show:
            ax.imshow(X[idx[i]].reshape(28, 28), cmap="gray")
            ax.set_title(f"Wahr: {y_true[idx[i]]} → Vorh.: {y_pred[idx[i]]}",
                        color="red", fontsize=10)
        ax.axis("off")
    plt.suptitle("Falsch klassifizierte Bilder", fontsize=14, color="red")
    plt.tight_layout()
    plt.show()


print("🔍 Fehleranalyse:")
n_errors = (y_test != y_pred).sum()
print(f"   {n_errors} von {len(y_test)} Testbildern falsch klassifiziert ({n_errors/len(y_test)*100:.2f}%)")
plot_misclassified(X_test, y_test, y_pred, n=10)

### 6.2 Fehler pro Ziffer
Welche Ziffern sind am schwierigsten zu erkennen?

In [ ]:
errors_per_digit = {}
for digit in range(10):
    mask = y_test == digit
    total = mask.sum()
    wrong = (y_test[mask] != y_pred[mask]).sum()
    errors_per_digit[digit] = wrong / total * 100

fig, ax = plt.subplots(figsize=(10, 5))
digits = list(errors_per_digit.keys())
rates = list(errors_per_digit.values())
bars = ax.bar(digits, rates, color=["green" if r < 5 else "orange" if r < 10 else "red" for r in rates], edgecolor="white")
ax.set_xlabel("Ziffer")
ax.set_ylabel("Fehlerrate (%)")
ax.set_title("Fehlerrate pro Ziffer")
for d, r in zip(digits, rates):
    ax.text(d, r + 0.3, f"{r:.1f}%", ha="center", fontsize=9)
ax.set_xticks(digits)
plt.tight_layout()
plt.show()

### 6.3 Häufigste Verwechslungen
Welche Ziffern-Paare werden am häufigsten verwechselt?

In [ ]:
cm = confusion_matrix(y_test, y_pred)
# Diagonale auf 0 setzen (korrekte Klassifikationen ignorieren)
np.fill_diagonal(cm, 0)

# Top-10 Verwechslungen finden
flat_indices = np.argsort(cm.flatten())[-10:][::-1]
rows, cols = np.unravel_index(flat_indices, cm.shape)

print("🔝 Top-10 häufigste Verwechslungen:")
print(f"   {'Wahr':>6s} → {'Vorh.':>6s}   {'Anzahl':>6s}")
print("   " + "-" * 30)
for r, c in zip(rows, cols):
    print(f"   {r:>6d} → {c:>6d}   {cm[r, c]:>6d}")

print("\n✅ Analyse abgeschlossen!")

## Zusammenfassung

In diesem Notebook haben wir:

1. ✅ Den **MNIST-Datensatz** geladen und exploriert (70.000 handgeschriebene Ziffern)
2. ✅ Ein **MLPClassifier**-Modell trainiert (784 → 128 → 64 → 10)
3. ✅ Die **Test-Accuracy** gemessen und eine **Confusion Matrix** erstellt
4. ✅ Eine **Fehleranalyse** durchgeführt — falsch klassifizierte Bilder visualisiert
5. ✅ Die **häufigsten Verwechslungen** identifiziert

**Nächste Schritte**:
- CNN mit PyTorch für höhere Accuracy trainieren
- Data Augmentation (Rotation, Verschiebung) ausprobieren
- Hyperparameter-Tuning (mehr Layer, andere Aktivierungsfunktionen)